#Imports

In [ ]:
!pip install woodelf_explainer==0.2.12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.8 MB/s eta 0:00:00


In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
from typing import Union, Dict, Optional, Tuple, Set, List
from math import factorial
import time
from copy import copy
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score
import scipy
import shap

import lightgbm as lgb

# For GPU execution
# import cupy as cp

In [ ]:
from woodelf.parse_models import load_decision_tree_ensemble_model
from woodelf.cube_metric import CubeMetric, ShapleyInteractionValues, ShapleyValues

import woodelf


In [ ]:
# Useful if you run this on google colab and downloaded the data into your drive.
# If you run the notebook in other environment remove these lines and change the 'pd.read_csv()' function in this notebook to read from
# where you saved you data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Environment Note
Use the T4 High-RAM colab runtime type. This is the T4 runtime type with 50GB of RAM.

# Fraud Data Preprocessing and Model training

In [ ]:
transactions_train = pd.read_parquet('drive/MyDrive/CURRENT_DIR/data/ieee_cis_fraud_train.parquet') # columns are train_features + ['isFraud']
transactions_test = pd.read_parquet('drive/MyDrive/CURRENT_DIR/data/ieee_cis_fraud_test.parquet') # columns are train_features + ['isFraud']

train_features = [f for f in transactions_train.columns if f != 'isFraud']
fraud_train = transactions_train[train_features]
fraud_test = transactions_test[train_features]

In [ ]:
LIGHTGBM_PARAMS = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.1,

    # Allow high depth and enough leaves to reach this possible depth
    "num_leaves": 2024,
    "max_depth": 10,
    "min_data_in_leaf": 500,        # Does provide some regulation

    # Sampling (stability + reduces overfit)
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,

    # Practical
    "verbosity": -1,
    "seed": 42,
    "force_col_wise": True,            # often faster/safer for wide data
}


def print_model_stat(model, features):
    mobj = load_decision_tree_ensemble_model(model, features)
    depths = {}
    for tree in mobj.trees:
        for leaf, path in tree.get_all_leaves_with_paths():
            depth = len(path)
            if depth not in depths:
                depths[depth] = 0
            depths[depth] += 1

    for depth in sorted(list(depths.keys())):
        print(f"\tPaths of depth {depth}: {depths.get(depth, 0)} paths")

def lightgbm_model(X_train, y_train, params, num_rounds=100):
    train_set = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
    return lgb.train(
        params=params,
        train_set=train_set,
        num_boost_round=num_rounds
    )


def different_depth_lightgbm_models(trainset, y, params, num_rounds, depths):
    models = {}
    for depth in depths:
        new_params = params.copy()
        new_params['max_depth'] = depth
        models[depth] = lightgbm_model(trainset, y, new_params, num_rounds=num_rounds)
        print("\n\n")
        print(f"Trained on depth {depth}")
        print_model_stat(models[depth], list(trainset.columns))
    return models

In [ ]:
gbm_100trees = different_depth_lightgbm_models(transactions_train[train_features], transactions_train['isFraud'], LIGHTGBM_PARAMS, num_rounds=100, depths=[6,9,12,15,18,21])




Trained on depth 6
	Paths of depth 1: 6 paths
	Paths of depth 2: 60 paths
	Paths of depth 3: 135 paths
	Paths of depth 4: 284 paths
	Paths of depth 5: 473 paths
	Paths of depth 6: 2086 paths



Trained on depth 9
	Paths of depth 1: 16 paths
	Paths of depth 2: 55 paths
	Paths of depth 3: 138 paths
	Paths of depth 4: 258 paths
	Paths of depth 5: 454 paths
	Paths of depth 6: 710 paths
	Paths of depth 7: 959 paths
	Paths of depth 8: 1222 paths
	Paths of depth 9: 3752 paths



Trained on depth 12
	Paths of depth 1: 15 paths
	Paths of depth 2: 55 paths
	Paths of depth 3: 143 paths
	Paths of depth 4: 290 paths
	Paths of depth 5: 472 paths
	Paths of depth 6: 636 paths
	Paths of depth 7: 884 paths
	Paths of depth 8: 1140 paths
	Paths of depth 9: 1496 paths
	Paths of depth 10: 1681 paths
	Paths of depth 11: 1991 paths
	Paths of depth 12: 4782 paths



Trained on depth 15
	Paths of depth 1: 16 paths
	Paths of depth 2: 72 paths
	Paths of depth 3: 121 paths
	Paths of depth 4: 278 paths
	Paths of

In [ ]:
def high_depth_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, background_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.high_depth_woodelf.woodelf_for_high_depth(model, consumer_data, background_data, metric=metric, global_importance=global_importance, GPU=GPU)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

# Background SHAP

In [ ]:
woodelf_running_times = high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_100trees[12], 15: gbm_100trees[15], 18: gbm_100trees[18], 21: gbm_100trees[21]},
    consumer_data=transactions_test[train_features], background_data=transactions_train[train_features], metric=ShapleyValues(),
    global_importance = False, GPU=True
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:17<00:00,  5.80it/s]


M time: 0.32 sec, s time: 11.08 sec (f prepare time: 4.900341987609863)
On Depth 6 Took: 17.959643602371216


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:43<00:00,  2.27it/s]


M time: 0.01 sec, s time: 33.64 sec (f prepare time: 13.996822118759155)
On Depth 9 Took: 44.456151485443115


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:38<00:00,  1.01it/s]


...

M time: 0.99 sec, s time: 139.87 sec (f prepare time: 58.170438289642334)
On Depth 15 Took: 170.63876748085022


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [04:25<00:00,  2.66s/it]


M time: 10.11 sec, s time: 231.14 sec (f prepare time: 87.52359080314636)
On Depth 18 Took: 277.31759905815125


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [10:52<00:00,  6.52s/it]

M time: 93.99 sec, s time: 601.34 sec (f prepare time: 134.23975348472595)
On Depth 21 Took: 747.8214235305786
Background SHAP: 17.959643602371216 & 44.456151485443115 & 99.4199869632721 & 170.63876748085022 & 277.31759905815125 & 747.8214235305786


# Background SHAP IV

In [ ]:
woodelf_running_times = high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_100trees[12], 15: gbm_100trees[15], 18: gbm_100trees[18]}, # depth 21 crashes due to RAM
    consumer_data=transactions_test[train_features].sample(10_000, random_state=42), background_data=transactions_train[train_features], metric=ShapleyInteractionValues(),
    global_importance = False, GPU=True
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:14<00:00,  7.03it/s]


M time: 0.0 sec, s time: 9.98 sec (f prepare time: 4.119342088699341)
On Depth 6 Took: 14.518475770950317


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:49<00:00,  2.04it/s]


M time: 0.02 sec, s time: 34.15 sec (f prepare time: 13.960595607757568)
On Depth 9 Took: 49.51755404472351


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:57<00:00,  1.17s/it]


M time: 0.26 sec, s time: 81.61 sec (f prepare time: 33.11275243759155)
On Depth 12 Took: 118.24374270439148


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [03:41<00:00,  2.22s/it]


M time: 3.55 sec, s time: 159.65 sec (f prepare time: 57.0676794052124)
On Depth 15 Took: 226.00175976753235


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [10:52<00:00,  6.52s/it]

M time: 48.71 sec, s time: 550.31 sec (f prepare time: 87.13130712509155)
On Depth 18 Took: 702.097517490387
Background SHAP: 14.518475770950317 & 49.51755404472351 & 118.24374270439148 & 226.00175976753235 & 702.097517490387


In [ ]:
# shap Package does not support Background SHAP IV

# RAM Crashes

In [ ]:
# Crashed due to RAM

woodelf_running_times = simple_woodelf_on_models_dict(
    {18: gbm_1trees[18]},
    consumer_data=transactions_test[train_features], background_data=transactions_train[train_features], metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Crashed due to RAM

woodelf_running_times = simple_woodelf_on_models_dict(
    {15: gbm_1trees[15]},
    consumer_data=transactions_test[train_features], background_data=transactions_train[train_features], metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Crashed due to RAM (in the HighDepthPathToMatrices.build_matrices phase)

woodelf_running_times = high_depth_woodelf_on_models_dict(
    {21: gbm_1trees[21]},
    consumer_data=transactions_test[train_features].sample(10_000, random_state=42), background_data=transactions_train[train_features], metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))